In [1]:
import random

# --------------------------------
# 1. Traffic signal configuration
# --------------------------------

PHASES = [
    "North-South Straight",
    "East-West Straight",
    "North-South Left",
    "East-West Left"
]

# Maximum queue lengths (vehicles)
queues = [27.4, 17.7, 5.7, 11.4]

NUM_PHASES = 4
POPULATION_SIZE = 50
GENERATIONS = 100
MUTATION_RATE = 0.15

# Traffic signal timing constraints
TOTAL_GREEN_TIME = 60
CLEARANCE_TIME = 4       # seconds per phase
MIN_GREEN = 8
MAX_GREEN = 30

TOTAL_CYCLE_TIME = (
    TOTAL_GREEN_TIME + NUM_PHASES * CLEARANCE_TIME
)

random.seed(42)


# --------------------------------
# 2. Generate a chromosome
# --------------------------------

def create_chromosome():
    timings = [
        random.randint(MIN_GREEN, MAX_GREEN)
        for _ in range(NUM_PHASES)
    ]

    return repair(timings)


# --------------------------------
# 3. Repair chromosome
# --------------------------------

def repair(chromosome):
    chromosome = [
        max(MIN_GREEN, min(MAX_GREEN, int(x)))
        for x in chromosome
    ]

    while sum(chromosome) != TOTAL_GREEN_TIME:

        difference = TOTAL_GREEN_TIME - sum(chromosome)

        if difference > 0:
            candidates = [
                i for i in range(NUM_PHASES)
                if chromosome[i] < MAX_GREEN
            ]
        else:
            candidates = [
                i for i in range(NUM_PHASES)
                if chromosome[i] > MIN_GREEN
            ]

        if not candidates:
            break

        i = random.choice(candidates)

        if difference > 0:
            chromosome[i] += 1
        else:
            chromosome[i] -= 1

    return chromosome


# --------------------------------
# 4. Fitness function
# --------------------------------

def calculate_delay(chromosome):

    # Lower delay means better traffic flow.
    delay = 0

    for i in range(NUM_PHASES):
        green = chromosome[i]
        queue = queues[i]

        delay += (queue ** 2) / green

    return delay * 2


def fitness(chromosome):
    # GA maximizes fitness, so minimize delay.
    return 1 / (1 + calculate_delay(chromosome))


# --------------------------------
# 5. Initialize population
# --------------------------------

population = [
    create_chromosome()
    for _ in range(POPULATION_SIZE)
]

# Include equal green allocation as a baseline
population[0] = [15, 15, 15, 15]

initial_best = min(
    calculate_delay(chromosome)
    for chromosome in population
)


# --------------------------------
# 6. Selection
# --------------------------------

def selection(population):

    candidates = random.sample(population, 3)

    return max(candidates, key=fitness)


# --------------------------------
# 7. Crossover
# --------------------------------

def crossover(parent1, parent2):

    point = random.randint(1, NUM_PHASES - 1)

    child = (
        parent1[:point] + parent2[point:]
    )

    return repair(child)


# --------------------------------
# 8. Mutation
# --------------------------------

def mutation(chromosome):

    if random.random() < MUTATION_RATE:

        i, j = random.sample(range(NUM_PHASES), 2)

        amount = random.randint(1, 3)

        if chromosome[i] - amount >= MIN_GREEN:
            if chromosome[j] + amount <= MAX_GREEN:
                chromosome[i] -= amount
                chromosome[j] += amount

    return chromosome


# --------------------------------
# 9. Genetic Algorithm
# --------------------------------

for generation in range(GENERATIONS):

    new_population = []

    # Elitism: preserve the best solution
    best = max(population, key=fitness)

    new_population.append(best[:])

    while len(new_population) < POPULATION_SIZE:

        parent1 = selection(population)
        parent2 = selection(population)

        child = crossover(parent1, parent2)

        child = mutation(child)

        new_population.append(child)

    population = new_population


# --------------------------------
# 10. Get optimal solution
# --------------------------------

best_solution = max(population, key=fitness)

optimal_delay = calculate_delay(best_solution)


# --------------------------------
# 11. Display output
# --------------------------------

print("\n========== TRAFFIC SIGNAL OPTIMIZATION ==========")

print(f"\nInitial best delay : {initial_best:.3f} seconds")

print(f"Optimized delay    : {optimal_delay:.3f} seconds")

print(f"\nTotal optimal cycle time : {TOTAL_CYCLE_TIME} seconds")

print("\n---------- OPTIMAL GREEN ALLOCATION ----------")

for i in range(NUM_PHASES):

    print(f"\n{PHASES[i]} : {best_solution[i]} sec")

    print(f"(Maximum queue : {queues[i]} vehicles)")

print("\n===============================================")


========== TRAFFIC SIGNAL OPTIMIZATION ==========

Initial best delay : 131.615 seconds
Optimized delay    : 130.974 seconds

Total optimal cycle time : 76 seconds

---------- OPTIMAL GREEN ALLOCATION ----------

North-South Straight : 25 sec
(Maximum queue : 27.4 vehicles)

East-West Straight : 16 sec
(Maximum queue : 17.7 vehicles)

North-South Left : 8 sec
(Maximum queue : 5.7 vehicles)

East-West Left : 11 sec
(Maximum queue : 11.4 vehicles)

